# FraudShild
### Realtime Fraud Detetcion Engine

## Part 1 : Data PreProcessing

### Imports and Data

In [1]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
# Set display options for better viewing in a notebook
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

In [3]:
train_tr = pd.read_csv("../data/raw/train_transaction.csv")
train_id = pd.read_csv("../data/raw/train_identity.csv")

test_tr = pd.read_csv("../data/raw/test_transaction.csv")
test_id = pd.read_csv("../data/raw/test_identity.csv")

### Quick Check on Data

In [4]:
print('train_transaction shape is {}'.format(train_tr.shape))
print('test_transaction shape is {}'.format(test_tr.shape))
print('train_identity shape is {}'.format(train_id.shape))
print('test_identity shape is {}'.format(test_id.shape))

train_transaction shape is (469572, 394)
test_transaction shape is (415814, 393)
train_identity shape is (144233, 41)
test_identity shape is (141907, 41)


In [ ]:
# input_file = '../data/raw/train_transaction.csv'
# output_file = '../data/mini/sample_fast.csv'
# sample_size = 20

# # 1. Count total rows in the file (efficiently)
# with open(input_file, 'r') as f:
#     # Sum 1 for every line in the file (subtract 1 for header)
#     total_rows = sum(1 for line in f) - 1

# print(f"Total rows in file: {total_rows}")

# # 2. Generate random indices to skip
# # We pick unique random numbers between 1 and total_rows
# skip_indices = sorted(random.sample(range(1, total_rows + 1), total_rows - sample_size))

# # 3. Read CSV while skipping the rows we didn't pick
# # logic: skiprows accepts a list of row numbers to IGNORE. 
# # We ignore everything EXCEPT our 20 target rows (and the header, which is row 0).
# df_sample = pd.read_csv(input_file, skiprows=skip_indices)

# # 4. Save
# df_sample.to_csv(output_file, index=False)

# print(f"Success! Created {output_file} with 20 random rows without loading the whole file.")

Total rows in file: 469572
Success! Created ../data/mini/sample_fast.csv with 20 random rows without loading the whole file.


In [5]:
train_tr['isFraud'].value_counts(normalize=True)

isFraud
0    0.964885
1    0.035115
Name: proportion, dtype: float64

### Merging the Datasets

In [6]:
train = train_tr.merge(train_id, on='TransactionID', how='left')
test = test_tr.merge(test_id, on='TransactionID', how='left')

In [7]:
# Clean up memory
del train_tr, train_id, test_tr, test_id

### Checking the Merged Data

In [18]:
# train.info()
# train.describe()
# train.isnull().sum().sort_values(ascending=False).head(20)

In [17]:
# print("\n--- Initial Data Check ---")

# # .info() is great for checking data types and non-null counts
# print("\nData Info:")
# # 'verbose=True' and 'show_counts=True' are important for wide dataframes
# train.info(verbose=True, show_counts=True)

# # .head() to see the first few rows
# print("\nData Head:")
# print(train.head())

# # .describe() for basic stats on numerical columns
# print("\nNumerical Describe:")
# print(train.describe())

# # Check our target variable 'isFraud'
# # This is our first look at the class imbalance
# print("\nTarget Variable 'isFraud' Distribution:")
# print(train['isFraud'].value_counts(normalize=True) * 100)

### Data Dictionary and Feature Categories

| Category             | Examples                              |
| -------------------- | ------------------------------------- |
| Transaction features | TransactionAmt, ProductCD             |
| Card features        | card1–card6                           |
| Address features     | addr1, addr2                          |
| Device features      | DeviceType, DeviceInfo                |
| ID features          | id_01–id_38                           |
| Categorical          | ProductCD, R_emaildomain, addr1/addr2 |
| Numerical            | TransactionAmt, V1–V339               |


### Categorical and Numerical Features
Creating seperate Dict for categorical features and numerical features

In [20]:
print("\n--- Column Dictionary ---")

# Our target variable
target = 'isFraud'

# 'TransactionID' is an identifier, not a feature
identifier = 'TransactionID'

# Find all numerical and categorical columns
# We'll classify columns as 'categorical' if their dtype is 'object'
# or if they are one of the known categorical groups (card, addr, M, etc.)

categorical_features = []
numerical_features = []

# Known categorical prefixes
cat_prefixes = ['ProductCD', 'card', 'addr', 'P_emaildomain', 'R_emaildomain', 'M', 'Device', 'id_']

for col in train.columns:
    if col == target or col == identifier:
        continue
    
    # Check if column name starts with any of the known prefixes
    is_cat = False
    for prefix in cat_prefixes:
        if col.startswith(prefix):
            categorical_features.append(col)
            is_cat = True
            break
            
    if not is_cat:
        # If not a known prefix, check dtype
        if train[col].dtype == 'object':
            categorical_features.append(col)
        else:
            numerical_features.append(col)

print(f"Found {len(numerical_features)} numerical features.")
print(f"Found {len(categorical_features)} categorical features.")

# You could store this in a dictionary
column_dictionary = {
    'target': target,
    'identifier': identifier,
    'numerical': numerical_features,
    'categorical': categorical_features
}

# print(f"Numerical: {numerical_features[:10]}...") # Uncomment to see
# print(f"Categorical: {categorical_features[:10]}...") # Uncomment to see


--- Column Dictionary ---


NameError: name 'train' is not defined

In [ ]:
# cat_cols = train.select_dtypes(include='object').columns
# train[cat_cols] = train[cat_cols].fillna("missing")

In [ ]:
# num_cols = train.select_dtypes(exclude='object').columns
# train[num_cols] = train[num_cols].fillna(-1)

In [ ]:
# gives fragmentation warning
# train['day'] = train['TransactionDT'] // (24*60*60)
# train['hour'] = (train['TransactionDT'] % (24*60*60)) // 3600

# this avoids fragmentation
# new_features = pd.DataFrame({
#     'day': train['TransactionDT'] // (24*60*60),
#     'hour': (train['TransactionDT'] % (24*60*60)) // 3600,
# })

# train = pd.concat([train, new_features], axis=1)

In [ ]:
# better approach when using multiple features

# feature_dict = {}

# feature_dict['day'] = train['TransactionDT'] // (24*60*60)
# feature_dict['hour'] = (train['TransactionDT'] % (24*60*60)) // 3600
# feature_dict['P_email_simp'] = train['P_emaildomain'].fillna('missing').str.split('.').str[0]
# feature_dict['DeviceType_cleaned'] = train['DeviceType'].fillna('unknown').str.lower()
# # ... add more engineered features here ...

# new_features = pd.DataFrame(feature_dict)
# train = pd.concat([train, new_features], axis=1)

#### Email domain features

In [ ]:
def simplify_email(x):
    if isinstance(x, str):
        return x.split('.')[0]
    return x

train['P_email_simp'] = train['P_emaildomain'].apply(simplify_email)
train['R_email_simp'] = train['R_emaildomain'].apply(simplify_email)

#### Device Features

* DeviceType
* DeviceInfo
* id_30, id_31, id_33 (OS, browser, resolution)

In [ ]:
train['DeviceInfo'] = train['DeviceInfo'].str.replace('NaN', 'unknown')
train['DeviceInfo'] = train['DeviceInfo'].fillna('unknown').str.lower()
train['DeviceType'] = train['DeviceType'].fillna('unknown').str.lower()

In [ ]:
train['OS'] = train['id_30'].fillna('unknown').str.split(' ').str[0].str.lower()

#### Address Features

In [ ]:
train['addr1'] = train['addr1'].fillna(-1).astype('int')
train['addr2'] = train['addr2'].fillna(-1).astype('int')

## EDA

In [21]:
# Set a style for plots
sns.set(style="whitegrid")

print("\n--- 1.7.1: Missing Value Analysis ---")

# Calculate missing value percentages
missing_values = (train.isnull().sum() / len(train_id)) * 100
missing_values = missing_values.sort_values(ascending=False)

# Store in a DataFrame for easy viewing
missing_df = pd.DataFrame({
    'column_name': missing_values.index,
    'missing_percentage': missing_values.values
})

print("Top 30 columns with the most missing values:")
print(missing_df.head(30))

# Let's see how many columns are, for example, > 50% missing
high_missing_cols = missing_df[missing_df['missing_percentage'] > 50]
print(f"\nNumber of columns with > 50% missing data: {len(high_missing_cols)}")

# --- Strategy for Missing Data ---
# A common strategy is to drop columns that are almost entirely empty.
# Let's set a threshold, e.g., 90%.
threshold = 90
cols_to_drop = missing_df[missing_df['missing_percentage'] > threshold]['column_name'].tolist()

print(f"\nColumns to drop (>{threshold}% missing): {len(cols_to_drop)}")
# print(cols_to_drop) # Uncomment to see the list

# Drop these columns from our dataframe and column dictionary
train_id = train_id.drop(columns=cols_to_drop)

# Update our column dictionary
column_dictionary['numerical'] = [c for c in column_dictionary['numerical'] if c not in cols_to_drop]
column_dictionary['categorical'] = [c for c in column_dictionary['categorical'] if c not in cols_to_drop]

print(f"\nNew dataframe shape after dropping columns: {train_id.shape}")


--- 1.7.1: Missing Value Analysis ---


NameError: name 'train' is not defined

## Processed Data Export

### Train Test Split

In [10]:
from sklearn.model_selection import train_test_split

# Define X (features) and y (target)
X = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud']

# Create your LOCAL Train and Test sets
# We use X_train for training/oversampling
# We use X_test for evaluation (your "Practice Exam")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Free up the massive 'train' variable now that we have split it
del train

### Feature Engineering

In [11]:
import pandas as pd
from sklearn.impute import SimpleImputer

# --- 1. Handling Missing Values (Statistical) ---

# Initialize the imputer
imputer = SimpleImputer(strategy='mean')

# FIT on TRAIN only
imputer.fit(X_train[['card1']]) 

# TRANSFORM both Train and Test
X_train['card1'] = imputer.transform(X_train[['card1']])
X_test['card1'] = imputer.transform(X_test[['card1']])

# --- 2. Frequency Encoding (Statistical) ---

# Calculate frequency map based on TRAIN only
freq_map = X_train['ProductCD'].value_counts(normalize=True)

# Map this logic to both Train and Test
X_train['ProductCD_freq'] = X_train['ProductCD'].map(freq_map)

# CRITICAL: If Test has a category not in Train, this creates NaNs.
# We map using the SAME freq_map derived from Train.
X_test['ProductCD_freq'] = X_test['ProductCD'].map(freq_map) 

# Handle new categories in Test that weren't in Train (fill with 0 or rare value)
X_test['ProductCD_freq'] = X_test['ProductCD_freq'].fillna(0)

print("Feature Engineering complete.")

Feature Engineering complete.


#### Time Feature

In [12]:
def engineering_time_features(df):
    df = df.copy() # Good practice to not modify original in place
    
    # 1. Convert TransactionDT (seconds) to meaningful time units
    # We assume the dataset starts on a random day at 00:00:00
    days = df['TransactionDT'] / (3600 * 24)
    
    # Get the hour of the day (0-23)
    df['hour'] = (df['TransactionDT'] // 3600) % 24
    
    # Get the day of the week (roughly)
    df['day'] = days // 1
    
    # 2. Cyclical Encoding for Hour (Optional but recommended)
    # 23:00 and 00:00 are close, but the numbers 23 and 0 are far. 
    # Sin/Cos encoding fixes this.
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    return df

print("Adding time features...")
X_train = engineering_time_features(X_train)
X_test = engineering_time_features(X_test)
print("Time features added.")

Adding time features...
Time features added.


#### Frequency Encoding

In [13]:
def frequency_encoding(train_df, test_df, columns):
    # Iterate through the columns we want to encode
    for col in columns:
        # 1. Calculate frequency map using ONLY Train data
        freq_map = train_df[col].value_counts(normalize=True)
        
        # 2. Map to Train
        # Suffix '_freq' to keep original column
        train_df[col + '_freq'] = train_df[col].map(freq_map)
        
        # 3. Map to Test
        test_df[col + '_freq'] = test_df[col].map(freq_map)
        
        # 4. Handle unseen values in Test (fill with 0)
        test_df[col + '_freq'] = test_df[col + '_freq'].fillna(0)
        
    return train_df, test_df

# Let's apply this to high-cardinality columns
cat_cols_to_encode = ['card1', 'card2', 'addr1', 'P_emaildomain']
print(f"Frequency encoding: {cat_cols_to_encode}.")
X_train, X_test = frequency_encoding(X_train, X_test, cat_cols_to_encode)

Frequency encoding: ['card1', 'card2', 'addr1', 'P_emaildomain'].


#### Mean Transaction Amount Per Card

In [14]:
def mean_target_encoding(train_df, test_df, group_col, target_col='TransactionAmt'):
    # 1. Calculate the mean Amount for each group (e.g., each card1) in TRAIN
    mean_map = train_df.groupby(group_col)[target_col].mean()
    
    new_col_name = f'{target_col}_div_mean_{group_col}'
    
    # 2. Map the mean back to the dataframe
    # For Train:
    train_mean = train_df[group_col].map(mean_map)
    # For Test: (Use the map from Train!)
    test_mean = test_df[group_col].map(mean_map)
    
    # 3. Create the feature: Interaction / Mean
    # "How much bigger is this transaction compared to the average for this card?"
    train_df[new_col_name] = train_df[target_col] / train_mean
    test_df[new_col_name] = test_df[target_col] / test_mean
    
    return train_df, test_df

print("Creating behavioral aggregations...")
# Check if current transaction is higher than average for this card type
X_train, X_test = mean_target_encoding(X_train, X_test, group_col='card1')
X_train, X_test = mean_target_encoding(X_train, X_test, group_col='card4') # card issuer (Visa/Mastercard)

Creating behavioral aggregations...


#### Handling Missing Values

In [15]:
# Identify numerical and categorical columns again (since we added new ones)
# Quick trick: Select object types for categorical, numbers for numerical
cat_cols = X_train.select_dtypes(include=['object']).columns
num_cols = X_train.select_dtypes(include=['number']).columns

# 1. Fill Categorical with "Unknown"
X_train[cat_cols] = X_train[cat_cols].fillna("Unknown")
X_test[cat_cols] = X_test[cat_cols].fillna("Unknown")

# 2. Fill Numerical with -999 
# (Common practice for Trees; it learns -999 is a special category)
X_train[num_cols] = X_train[num_cols].fillna(-999)
X_test[num_cols] = X_test[num_cols].fillna(-999)

print(f"Feature Engineering Complete. Train shape: {X_train.shape}")

Feature Engineering Complete. Train shape: (375657, 443)


#### Lable Encoding Remaining Features

In [16]:
from sklearn.preprocessing import OrdinalEncoder

print("--- Converting Categorical Strings to Numbers ---")

# 1. Identify all Categorical Columns (Object type)
# We must ensure we only grab the columns that are still objects
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Encoding {len(cat_cols)} columns: {cat_cols[:5]}... and more.")

# 2. Initialize Ordinal Encoder
# handle_unknown='use_encoded_value' -> tells it not to crash on new labels
# unknown_value=-1 -> tells it to replace new labels with -1
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# 3. Fit on Train
encoder.fit(X_train[cat_cols])

# 4. Transform Train and Test
# We cast to float32 to save memory (default is float64)
X_train[cat_cols] = encoder.transform(X_train[cat_cols]).astype('float32')
X_test[cat_cols] = encoder.transform(X_test[cat_cols]).astype('float32')

print("Encoding complete.")

# Final Check: Ensure all columns are numeric
# If this prints anything other than "0 non-numeric columns", we have a problem.
non_numeric = X_train.select_dtypes(exclude=['number']).columns
print(f"Remaining non-numeric columns: {len(non_numeric)}")

--- Converting Categorical Strings to Numbers ---
Encoding 31 columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']... and more.
Encoding complete.
Remaining non-numeric columns: 0


### Handling Class Imbalance

In [22]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print("--- 1.9: Handling Class Imbalance (SMOTE) ---")

# 1. Check distribution BEFORE SMOTE
print(f"Original Training Target Stats: {Counter(y_train)}")
# Output example: {0: 100000, 1: 3500} (Highly imbalanced)

# 2. Initialize SMOTE
# sampling_strategy=0.1 means we want the minority class (fraud) 
# to be 10% the size of the majority class. 
# You can use 'auto' to make them equal (50/50), but that creates HUGE data
# and often leads to overfitting. 0.1 or 0.2 is usually a safer sweet spot.
smote = SMOTE(sampling_strategy=0.1, random_state=42)

print("Resampling training data... (this may take a moment)")

# 3. Fit and Resample (ONLY on Train)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 4. Check distribution AFTER SMOTE
print(f"Resampled Training Target Stats: {Counter(y_train_resampled)}")

print(f"Original Train Shape: {X_train.shape}")
print(f"New Resampled Train Shape: {X_train_resampled.shape}")

# Clean up
# We can now replace the old X_train with the resampled version to save memory
del X_train, y_train
X_train = X_train_resampled
y_train = y_train_resampled

print("SMOTE applied successfully.")

--- 1.9: Handling Class Imbalance (SMOTE) ---
Original Training Target Stats: Counter({0: 362466, 1: 13191})
Resampling training data... (this may take a moment)
Resampled Training Target Stats: Counter({0: 362466, 1: 36246})
Original Train Shape: (375657, 443)
New Resampled Train Shape: (398712, 443)
SMOTE applied successfully.


### Checkpointing

In [23]:
print("--- Checkpointing Data ---")

# 1. Create a directory to keep things organized
os.makedirs('../data/processed', exist_ok=True)

# 2. Save the DataFrames/Series to Pickle files
# We use pickle because it preserves the exact Python object structure (dtypes, index, etc.)
print("Saving X_train...")
X_train.to_pickle('../data/processed/X_train_engineered.pkl')

print("Saving X_test...")
X_test.to_pickle('../data/processed/X_test_engineered.pkl')

print("Saving y_train...")
y_train.to_pickle('../data/processed/y_train_engineered.pkl')

print("Saving y_test...")
y_test.to_pickle('../data/processed/y_test_engineered.pkl')

print("Checkpoint saved successfully to '../data/processed/'")

--- Checkpointing Data ---
Saving X_train...
Saving X_test...
Saving y_train...
Saving y_test...
Checkpoint saved successfully to '../data/processed/'


#### Saving the Encoder

In [24]:
import joblib
import json
# 3. Save the Ordinal Encoder
# We need this to handle the categorical columns that weren't frequency encoded
joblib.dump(encoder, "ordinal_encoder.joblib")
print("Ordinal Encoder saved.")
# moved this from notebook1


Ordinal Encoder saved.


In [ ]:
# 1. Identify columns that used frequency encoding
# (Must match the ones we did in step 1.6)
freq_cols = ['card1', 'card2', 'addr1', 'P_emaildomain']

# 2. Compute and save the maps
freq_maps = {}
for col in freq_cols:
    # We convert the series to a dict so JSON can save it
    freq_maps[col] = X_train[col].value_counts(normalize=True).to_dict()

# 3. Save to file
with open("frequency_maps.json", "w") as f:
    json.dump(freq_maps, f)

print("Frequency maps saved.")

Frequency maps saved.


: 